# Phase 7: Hint Generator Experiments

This notebook demonstrates the hint generation pipeline for Model B. We train a hint ranker to score sentences from articles and select three graduated hints for reading comprehension questions.

## Contents
1. Setup and Imports
2. Data Loading
3. Feature Analysis
4. Model Training
5. Evaluation Metrics
6. Sample Hint Generation

## 1. Setup and Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

# Add src to path
sys.path.insert(0, str(Path.cwd() / 'src'))

from preprocessing import load_raw_splits, clean_text
from hint_generator import (
    split_into_sentences,
    compute_sentence_features_batch,
    label_gold_hint_sentence,
    generate_hints,
    load_hint_model,
)

# Set random seed for reproducibility
np.random.seed(42)

print("Setup complete!")

: 

## 2. Data Loading

In [ ]:
# Load raw splits
raw_splits = load_raw_splits(Path('data/raw'))

train_df = raw_splits['train'].copy()
dev_df = raw_splits['dev'].copy()
test_df = raw_splits['test'].copy()

print(f"Train: {len(train_df)} samples")
print(f"Dev: {len(dev_df)} samples")
print(f"Test: {len(test_df)} samples")

# Display sample
print("\nSample from training data:")
sample = train_df.iloc[0]
print(f"Article (first 200 chars): {str(sample['article'])[:200]}...")
print(f"Question: {sample['question']}")
print(f"Correct Answer: {sample[sample['answer']]}")

## 3. Feature Analysis

In [ ]:
# Analyze sentences and features for a sample
sample_idx = 5
sample = train_df.iloc[sample_idx]

article = str(sample['article'])
question = str(sample['question'])
correct_answer = str(sample[sample['answer']])

# Split sentences
sentences = split_into_sentences(article)
print(f"Article split into {len(sentences)} sentences\n")

# Show first 3 sentences
print("First 3 sentences:")
for i, sent in enumerate(sentences[:3]):
    print(f"  {i}: {sent[:80]}...")

print(f"\nQuestion: {question}")
print(f"Correct Answer: {correct_answer}")

# Find gold label
gold_idx = label_gold_hint_sentence(sentences, article, correct_answer)
print(f"\nGold hint sentence index: {gold_idx}")
print(f"Gold hint: {sentences[gold_idx][:100]}...")

In [ ]:
# Compute and visualize feature statistics
from preprocessing import build_vectorizer

# Build vectorizer
vocab_texts = []
vocab_texts.extend(train_df['article'].astype(str).tolist()[:1000])  # Sample for speed
vocab_texts.extend(train_df['question'].astype(str).tolist()[:1000])

vectorizer = build_vectorizer(max_features=5000, min_df=2)
vectorizer.fit(vocab_texts)

# Compute features
X = compute_sentence_features_batch(sentences, article, question, correct_answer, vectorizer)

# Feature names
feature_names = [
    'Word overlap with Q',
    'Word overlap with A',
    'Position ratio',
    'Sentence length',
    'Named entity proxy',
    'WH cue',
    'Keyword density',
    'Distance to answer'
]

print("Feature statistics for sample:")
print(pd.DataFrame(X, columns=feature_names).describe())

# Show features for gold sentence
print(f"\nFeatures for gold hint sentence (index {gold_idx}):")
print(pd.Series(X[gold_idx], index=feature_names))

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feature_name in enumerate(feature_names):
    ax = axes[i]
    ax.hist(X[:, i], bins=20, alpha=0.7, edgecolor='black')
    ax.axvline(X[gold_idx, i], color='red', linestyle='--', linewidth=2, label='Gold')
    ax.set_xlabel(feature_name)
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('models/model_b/hint_generator/feature_distributions.png', dpi=100)
plt.show()

print("Feature distribution plot saved!")

## 4. Model Training

The hint model is trained using LogisticRegression to rank sentences by relevance. Training has been completed and the model is saved to `models/model_b/hint_generator/`.

In [ ]:
# Load trained hint model
try:
    hint_ranker, hint_vectorizer = load_hint_model(Path('models/model_b/hint_generator'))
    print("✓ Hint model loaded successfully!")
    print(f"Model type: {type(hint_ranker).__name__}")
    print(f"Vectorizer type: {type(hint_vectorizer).__name__ if hint_vectorizer else 'None'}")
except Exception as e:
    print(f"⚠ Could not load hint model: {e}")
    print("Note: Run 'python src/model_b_train.py --train-hints' to train the model.")

## 5. Evaluation Metrics

In [ ]:
# Summary of evaluation metrics
print("="*60)
print("HINT GENERATOR EVALUATION SUMMARY")
print("="*60)
print()
print("Metrics computed during training:")
print()
print("Dev Set:")
print("  Precision@1: Fraction of top-1 predictions matching gold hint")
print("  Precision@3: Fraction of gold hints appearing in top-3")
print("  F1: Binary classification F1 score")
print("  Accuracy: Binary classification accuracy")
print()
print("Test Set:")
print("  Same metrics as dev set")
print()
print("For detailed metrics, see training output or logs.")

## 6. Sample Hint Generation

## 5.5 Confusion Matrix

In [ ]:
# Function to display hints nicely
def display_hints(article, question, correct_answer, hints):
    print("="*70)
    print("Article excerpt:")
    print(article[:300] + "..." if len(article) > 300 else article)
    print()
    print(f"Question: {question}")
    print(f"Correct Answer: {correct_answer}")
    print()
    print("Generated Hints (graduated from general to specific):")
    print(f"  Hint 1 (General): {hints['hint_1']}")
    print()
    print(f"  Hint 2 (Specific): {hints['hint_2']}")
    print()
    print(f"  Hint 3 (Strong):   {hints['hint_3']}")
    print("="*70)

print("Sample hint generation function defined.")

In [ ]:
# Generate hints for multiple examples
if 'hint_ranker' in locals():
    print("Generating sample hints...\n")
    
    for idx in range(min(3, len(test_df))):
        sample = test_df.iloc[idx]
        article = str(sample['article'])
        question = str(sample['question'])
        correct_answer = str(sample[sample['answer']])
        
        hints = generate_hints(
            article, question, correct_answer, hint_ranker, hint_vectorizer
        )
        
        display_hints(article, question, correct_answer, hints)
        print()
else:
    print("Hint model not loaded. Cannot generate hints.")
    print("Please train the model first: python src/model_b_train.py --train-hints")

## 7. Model B Complete Pipeline

After Phase 7, Model B provides:
- **3 Distractors**: Generated using the distractor ranker
- **3 Graduated Hints**: Generated using the hint generator
- **Correct Answer**: From the original dataset

This gives a complete experience for users engaging with reading comprehension questions.

In [ ]:
# Example of complete Model B output
if 'hint_ranker' in locals():
    print("MODEL B COMPLETE OUTPUT EXAMPLE")
    print("="*70)
    
    sample = test_df.iloc[0]
    article = str(sample['article'])
    question = str(sample['question'])
    correct_answer = str(sample[sample['answer']])
    
    hints = generate_hints(
        article, question, correct_answer, hint_ranker, hint_vectorizer
    )
    
    print(f"Question: {question}")
    print()
    print(f"Correct Answer: {correct_answer}")
    print()
    print("Graduated Hints for Learner:")
    print(f"  1. {hints['hint_1']}")
    print(f"  2. {hints['hint_2']}")
    print(f"  3. {hints['hint_3']}")
    print()
    print("Note: Distractors would be generated separately by the distractor ranker")
else:
    print("Model not loaded.")